In [3]:
print("Hello")

Hello


In [4]:
!pip install pandas openpyxl

Defaulting to user installation because normal site-packages is not writeable


In [5]:
import pandas as pd
df = pd.read_excel(r"C:\Users\shaik\Desktop\Project\archive\online_retail_II.xlsx")
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [6]:
df.shape

(525461, 8)

In [7]:
df.isnull().sum()

Invoice             0
StockCode           0
Description      2928
Quantity            0
InvoiceDate         0
Price               0
Customer ID    107927
Country             0
dtype: int64

In [8]:
df[df['Quantity'] < 0]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
...,...,...,...,...,...,...,...,...
525231,538159,21324,NaN,-18,2010-12-09 17:17:00,0.00,NaN,United Kingdom
525232,538158,20892,NaN,-32,2010-12-09 17:17:00,0.00,NaN,United Kingdom
525234,538161,46000S,Dotcom sales,-100,2010-12-09 17:25:00,0.00,NaN,United Kingdom
525235,538162,46000M,Dotcom sales,-100,2010-12-09 17:25:00,0.00,NaN,United Kingdom


In [9]:
df['InvoiceDate'].min(), df['InvoiceDate'].max()

(Timestamp('2009-12-01 07:45:00'), Timestamp('2010-12-09 20:01:00'))

In [10]:
df = df.dropna(subset=['Customer ID'])

In [11]:
df.shape

(417534, 8)

In [12]:
df = df[df['Quantity'] > 0]

In [13]:
df.shape

(407695, 8)

In [14]:
df.duplicated().sum()

np.int64(6748)

In [15]:
df = df.drop_duplicates()

In [16]:
df.shape

(400947, 8)

In [17]:
df['Revenue'] = df['Quantity'] * df['Price']

In [18]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0


In [19]:
customer_df = df.groupby('Customer ID').agg({
    'InvoiceDate': 'max',
    'Invoice': 'nunique',
    'Revenue': 'sum'
}).reset_index()

In [20]:
customer_df.columns = [
    'CustomerID',
    'LastPurchaseDate',
    'TotalTransactions',
    'TotalRevenue'
]

In [21]:
customer_df.head()

,CustomerID,LastPurchaseDate,TotalTransactions,TotalRevenue
0,12346.0,2010-06-28 13:53:00,11,372.86
1,12347.0,2010-12-07 14:57:00,2,1323.32
2,12348.0,2010-09-27 14:59:00,1,222.16
3,12349.0,2010-10-28 08:23:00,3,2671.14
4,12351.0,2010-11-29 15:23:00,1,300.93


In [22]:
customer_df.shape

(4314, 4)

In [23]:
reference_date = df['InvoiceDate'].max()

In [24]:
customer_df['Recency'] = (reference_date - customer_df['LastPurchaseDate']).dt.days

In [25]:
customer_df.shape

(4314, 5)

In [26]:
customer_df.head()

,CustomerID,LastPurchaseDate,TotalTransactions,TotalRevenue,Recency
0,12346.0,2010-06-28 13:53:00,11,372.86,164
1,12347.0,2010-12-07 14:57:00,2,1323.32,2
2,12348.0,2010-09-27 14:59:00,1,222.16,73
3,12349.0,2010-10-28 08:23:00,3,2671.14,42
4,12351.0,2010-11-29 15:23:00,1,300.93,10


In [27]:
customer_df['Frequency'] = customer_df['TotalTransactions']
customer_df['Monetary'] = customer_df['TotalRevenue']

In [28]:
customer_df = customer_df[['CustomerID', 'Recency', 'Frequency', 'Monetary']]

In [29]:
customer_df.head()

,CustomerID,Recency,Frequency,Monetary
0,12346.0,164,11,372.86
1,12347.0,2,2,1323.32
2,12348.0,73,1,222.16
3,12349.0,42,3,2671.14
4,12351.0,10,1,300.93


In [30]:
customer_df['R_score'] = pd.qcut(customer_df['Recency'], 4, duplicates='drop')
customer_df['F_score'] = pd.qcut(customer_df['Frequency'], 4, duplicates='drop')
customer_df['M_score'] = pd.qcut(customer_df['Monetary'], 4, duplicates='drop')

In [31]:
customer_df.head()

,CustomerID,Recency,Frequency,Monetary,R_score,F_score,M_score
0,12346.0,164,11,372.86,"(135.0, 373.0]","(5.0, 205.0]","(307.105, 700.405]"
1,12347.0,2,2,1323.32,"(-0.001, 17.0]","(0.999, 2.0]","(700.405, 1713.298]"
2,12348.0,73,1,222.16,"(52.0, 135.0]","(0.999, 2.0]","(-0.001, 307.105]"
3,12349.0,42,3,2671.14,"(17.0, 52.0]","(2.0, 5.0]","(1713.298, 349164.35]"
4,12351.0,10,1,300.93,"(-0.001, 17.0]","(0.999, 2.0]","(-0.001, 307.105]"


In [32]:
customer_df['R_score'] = pd.qcut(customer_df['Recency'], 4, duplicates='drop').cat.codes + 1
customer_df['F_score'] = pd.qcut(customer_df['Frequency'], 4, duplicates='drop').cat.codes + 1
customer_df['M_score'] = pd.qcut(customer_df['Monetary'], 4, duplicates='drop').cat.codes + 1

In [33]:
customer_df.head()

,CustomerID,Recency,Frequency,Monetary,R_score,F_score,M_score
0,12346.0,164,11,372.86,4,3,2
1,12347.0,2,2,1323.32,1,1,3
2,12348.0,73,1,222.16,3,1,1
3,12349.0,42,3,2671.14,2,2,4
4,12351.0,10,1,300.93,1,1,1


In [34]:
customer_df['RFM_Score'] = (
    customer_df['R_score'].astype(str) +
    customer_df['F_score'].astype(str) +
    customer_df['M_score'].astype(str)
)

In [35]:
customer_df.head()

,CustomerID,Recency,Frequency,Monetary,R_score,F_score,M_score,RFM_Score
0,12346.0,164,11,372.86,4,3,2,432
1,12347.0,2,2,1323.32,1,1,3,113
2,12348.0,73,1,222.16,3,1,1,311
3,12349.0,42,3,2671.14,2,2,4,224
4,12351.0,10,1,300.93,1,1,1,111


In [36]:
def segment_customer(row):
    if row['R_score'] >= 3 and row['F_score'] >= 3:
        return 'High Value'
    elif row['R_score'] >= 2 and row['F_score'] >= 2:
        return 'Medium Value'
    else:
        return 'Low Value'

customer_df['Segment'] = customer_df.apply(segment_customer, axis=1)

In [37]:
customer_df['Segment'].value_counts()

Segment
Low Value       3070
Medium Value    1087
High Value       157
Name: count, dtype: int64

In [38]:
customer_df['Churn'] = customer_df['Recency'].apply(lambda x: 1 if x > 90 else 0)

In [39]:
customer_df['Churn'].value_counts()

Churn
0    2885
1    1429
Name: count, dtype: int64

In [40]:
churned_revenue = customer_df[customer_df['Churn'] == 1]['Monetary'].sum()
total_revenue = customer_df['Monetary'].sum()

churned_revenue, total_revenue

(np.float64(1073406.464), np.float64(8798233.743999999))

In [41]:
(churned_revenue / total_revenue) * 100

np.float64(12.200249450431059)

In [43]:
df.to_csv("cleaned_data.csv", index=False)

In [44]:
import os
os.getcwd()

'C:\\Users\\shaik'